# Module 07: Weather Ground-Stop Classifier (RandomForest)

Predicts the probability of a Cat III ILS ground stop from live METAR-style visibility/ceiling/wind, trained on real VABO weather history from the IEM ASOS archive. This is the single biggest driver of `core/models.py`'s composite per-flight risk score.

In [ ]:
!pip install -q scikit-learn==1.5.2 pandas==2.2.3 numpy==1.26.4 joblib==1.4.2 requests==2.32.3


## 1. Fetch real METAR history for VABO from the Iowa Environmental Mesonet

The IEM ASOS/METAR archive covers global airport weather stations, including
India (`network=IN__ASOS`), via a single documented HTTP GET with no
authentication - see memory.md's linked source. We pull ~2 years of hourly
observations for Vadodara and derive a binary "low-visibility ground-stop
risk" label from the reported visibility/ceiling, matching how a Cat III ILS
ground-stop decision is actually made.

In [ ]:
import io
import numpy as np
import pandas as pd
import requests

IEM_URL = (
    "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"
    "?station=VABO&network=IN__ASOS"
    "&data=vsby,skyl1,sknt"
    "&year1=2023&month1=1&day1=1&year2=2025&month2=1&day2=1"
    "&tz=UTC&format=onlycomma&latlon=no&missing=M&trace=T&direct=no"
)

def parse_metar_csv(text):
    df = pd.read_csv(io.StringIO(text))
    df.columns = [c.strip().lower() for c in df.columns]
    for col in ("vsby", "skyl1", "sknt"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["vsby", "sknt"])
    if len(df) < 100:
        raise ValueError("Too few valid METAR rows returned for VABO.")
    return df

try:
    resp = requests.get(IEM_URL, timeout=30)
    resp.raise_for_status()
    metar = parse_metar_csv(resp.text)

    visibility_m = (metar["vsby"] * 1609.34).clip(0, 16000)          # statute miles -> meters
    ceiling_ft = metar["skyl1"].fillna(3000).clip(0, 12000)          # feet, missing = high/clear
    wind_kt = metar["sknt"].clip(0, 60)

    df = pd.DataFrame({"visibility_m": visibility_m, "ceiling_ft": ceiling_ft, "wind_kt": wind_kt})
    # Cat III ground-stop proxy: very low visibility OR very low ceiling
    df["ground_stop"] = ((df["visibility_m"] < 1000) | (df["ceiling_ft"] < 200)).astype(int)
    data_source = f"IEM ASOS/METAR archive for VABO, IN__ASOS network ({len(df)} observations, live download)"

except Exception as exc:
    print(f"[fallback] IEM METAR archive unreachable for VABO ({exc}); generating synthetic METAR-like data.")
    rng = np.random.default_rng(3)
    n = 6000
    # Bimodal mixture: ~90% normal operating conditions, ~10% winter-fog / monsoon-haze
    # episodes, matching Gujarat's real Cat III fog-season climatology (Dec-Jan mornings).
    is_fog_episode = rng.random(n) < 0.10
    visibility_m = np.where(
        is_fog_episode,
        rng.gamma(shape=2.0, scale=350, size=n).clip(50, 3000),
        rng.gamma(shape=6.0, scale=1500, size=n).clip(1500, 10000),
    )
    ceiling_ft = np.where(
        is_fog_episode,
        rng.gamma(shape=2.0, scale=150, size=n).clip(50, 1500),
        rng.gamma(shape=5.0, scale=700, size=n).clip(800, 6000),
    )
    wind_kt = rng.normal(9, 6, n).clip(0, 45)
    df = pd.DataFrame({"visibility_m": visibility_m, "ceiling_ft": ceiling_ft, "wind_kt": wind_kt})
    df["ground_stop"] = ((df["visibility_m"] < 1000) | (df["ceiling_ft"] < 200)).astype(int)
    data_source = "synthetic METAR-like data (Gujarat winter-fog / monsoon climatology)"

print(f"Rows: {len(df)} | ground-stop rate: {df['ground_stop'].mean():.2%} | source: {data_source}")
df.head()


## 2. Train + evaluate the RandomForest ground-stop classifier

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = df[["visibility_m", "ceiling_ft", "wind_kt"]]
y = df["ground_stop"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() > 1 else None
)

clf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)

print(classification_report(y_test, clf.predict(X_test), zero_division=0))


## 3. Export for the live twin

In [ ]:
import joblib
from datetime import datetime, timezone

PKL_NAME = "07_weather_ground_stop.pkl"
joblib.dump(
    {
        "model": clf,
        "features": ["visibility_m", "ceiling_ft", "wind_kt"],
        "trained_at": datetime.now(timezone.utc).isoformat(),
        "data_source": data_source,
        "module": "07_weather_ground_stop",
    },
    PKL_NAME,
)
print(f"Saved {PKL_NAME}")


In [ ]:
# --- Download the trained artifact (Colab only; safe to run locally too) ---
try:
    from google.colab import files
    files.download(PKL_NAME)
    print(f"Downloading {PKL_NAME} ... move it into core/models/ on your machine.")
except ImportError:
    print(f"Not running in Colab - {PKL_NAME} is already saved in the current directory.")
    print("Copy it into core/models/ on your machine to activate this module in the live twin.")
